## **SCD Type1**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.default.scdtype1_source
(
    product_id INT PRIMARY KEY,
    product_name STRING,
    product_category STRING,
    process_date DATE
)

In [0]:
%sql
INSERT INTO datamodeling.default.scdtype1_source
VALUES
(1, 'Product1', 'Category1', CURRENT_DATE()),
(2, 'Product2', 'Category2', CURRENT_DATE()),
(3, 'Product3', 'Category3', CURRENT_DATE())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM datamodeling.default.scdtype1_source

product_id,product_name,product_category,process_date
1,Product1,Category1,2026-05-06
2,Product2,Category2,2026-05-06
3,Product3,NEW_Category6,2026-05-06


In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.scdtype1_table
(
    product_id INT PRIMARY KEY,
    product_name STRING,
    product_category STRING,
    process_date DATE
)

In [0]:
spark.sql("SELECT * FROM datamodeling.default.scdtype1_source").createOrReplaceTempView("src")

In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype1_table AS trg
USING src
ON trg.product_id = src.product_id

WHEN MATCHED AND src.process_date >= trg.process_date THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtype1_table

product_id,product_name,product_category,process_date
1,Product1,Category1,2026-05-06
2,Product2,Category2,2026-05-06
3,Product3,NEW_Category6,2026-05-06


In [0]:
%sql
UPDATE datamodeling.default.scdtype1_source
SET product_category = 'NEW_Category6'
WHERE product_id = 3

num_affected_rows
1


## **SCD Type2**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.default.scdtype2_source
(
    product_id INT PRIMARY KEY,
    product_name STRING,
    product_category STRING,
    process_date DATE
)

In [0]:
%sql
INSERT INTO datamodeling.default.scdtype2_source
VALUES
(1, 'Product1', 'Category1', CURRENT_DATE()),
(2, 'Product2', 'Category2', CURRENT_DATE()),
(3, 'Product3', 'Category3', CURRENT_DATE())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * ,current_timestamp AS start_date, 
CAST('4000-01-01' AS timestamp) AS end_date, 'Y' AS available
FROM datamodeling.default.scdtype2_source

product_id,product_name,product_category,process_date,start_date,end_date,available
1,Product1,Category1,2026-05-06,2026-05-06T10:11:51.810Z,4000-01-01T00:00:00.000Z,Y
2,Product2,Category2,2026-05-06,2026-05-06T10:11:51.810Z,4000-01-01T00:00:00.000Z,Y
3,Product3,Category3,2026-05-06,2026-05-06T10:11:51.810Z,4000-01-01T00:00:00.000Z,Y


In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.scdtype2_table
(
    product_id INT PRIMARY KEY,
    product_name STRING,
    product_category STRING,
    process_date DATE,
    start_date DATE,
    end_date DATE,
    available STRING
)

In [0]:
spark.sql("""SELECT * ,current_timestamp AS start_date, 
CAST('4000-01-01' AS timestamp) AS end_date, 'Y' AS available
FROM datamodeling.default.scdtype2_source""").createOrReplaceTempView("src2")

In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype2_table AS trg
USING src2
ON trg.product_id = src2.product_id AND trg.available = 'Y' 

-- When New DATA Updates
WHEN MATCHED AND (
    src2.product_name != trg.product_name OR
    src2.product_category != trg.product_category OR
    src2.process_date != trg.process_date
)THEN UPDATE SET 
        trg.end_date = current_timestamp(), trg.available = 'N'

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtype2_table

product_id,product_name,product_category,process_date,start_date,end_date,available
1,Product1,Category1,2026-05-06,2026-05-06,4000-01-01,Y
2,Product2,Category2,2026-05-06,2026-05-06,4000-01-01,Y
3,Product3,Category3,2026-05-06,2026-05-06,2026-05-06,N


In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype2_table AS trg
USING src2
ON trg.product_id = src2.product_id AND trg.available = 'Y' 

WHEN NOT MATCHED THEN INSERT(
    product_id,
    product_name,
    product_category,
    process_date,
    start_date,
    end_date,
    available
) VALUES(
    src2.product_id,
    src2.product_name,
    src2.product_category,
    src2.process_date,
    src2.start_date,
    src2.end_date,
    src2.available
)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtype2_table

product_id,product_name,product_category,process_date,start_date,end_date,available
1,Product1,Category1,2026-05-06,2026-05-06,4000-01-01,Y
2,Product2,Category2,2026-05-06,2026-05-06,4000-01-01,Y
3,Product3,Category3,2026-05-06,2026-05-06,2026-05-06,N
3,Product3,NEW_Category6,2026-05-06,2026-05-06,4000-01-01,Y


In [0]:
%sql
UPDATE datamodeling.default.scdtype2_source
SET product_category = 'NEW_Category6'
WHERE product_id = 3

num_affected_rows
1
